# 02 · Klasifikasi Hujan — Bab 3

*Pengantar Deep Learning untuk Meteorologi* · Kanada Kurniawan

Notebook pendamping Bab 3: membangun model klasifikasi biner (hujan/tidak hujan) dan mempelajari precision/recall/F1 serta trade-off threshold.

Prasyarat: Bab 1 (`ch-01-00_fondasi_tensorflow`) dan Bab 2 (`ch-02-01_regresi_pasang_surut`).

## 1. Setup & Verifikasi Lingkungan

In [ ]:
import tensorflow as tf
import numpy as np

print("TensorFlow:", tf.__version__)
print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

## 2. Data Sintetik Tidak Seimbang

Kita buat data sederhana meniru prediksi hujan deras (>50 mm) yang jarang terjadi: mayoritas 'tidak hujan deras', beberapa 'hujan deras'.

In [ ]:
np.random.seed(42)

n = 2000
# Fitur: [kelembapan, tekanan] ~ dua kelompok
x = np.random.randn(n, 2)
prob_deras = 0.05  # hanya 5% hujan deras
y = (np.random.rand(n) < (prob_deras + 0.4 * (x[:,0] > 1))).astype(float)

print("Distribusi label:")
print("  Hujan deras:", int(y.sum()), "| Tidak hujan deras:", int((1-y).sum()))

## 3. Split Berbasis Waktu

Meski sintetik, kita tetap memakai split urutan (bukan acak) untuk konsistensi dengan data deret waktu nyata.

In [ ]:
n_train = int(n * 0.7)
n_val = int(n * 0.15)

X_train, y_train = x[:n_train], y[:n_train]
X_val, y_val = x[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test, y_test = x[n_train+n_val:], y[n_train+n_val:]
print(f"train {X_train.shape} | val {X_val.shape} | test {X_test.shape}")

## 4. Model Klasifikasi Biner

MLP dengan lapisan keluaran **sigmoid** dan loss `binary_crossentropy`. Kita pantau precision & recall langsung saat training.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(8, activation="relu", input_shape=(2,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.Precision(), tf.keras.metrics.Recall()],
)
model.summary()

In [ ]:
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50, batch_size=32, verbose=0,
)
print("Metric terakhir (val):")
for k in history.history:
    if 'val_' in k:
        print(f"  {k}: {history.history[k][-1]:.4f}")

## 5. Akurasi vs Metrik pada Data Tidak Seimbang

Hitung metrik pada threshold berbeda (0.2, 0.5, 0.8) untuk melihat trade-off.

In [ ]:
def hitung_metrik(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tp = int(np.sum((y_pred == 1) & (y_true == 1)))
    fp = int(np.sum((y_pred == 1) & (y_true == 0)))
    fn = int(np.sum((y_pred == 0) & (y_true == 1)))
    tn = int(np.sum((y_pred == 0) & (y_true == 0)))
    prec = tp / (tp + fp) if (tp + fp) else 0
    rec = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    acc = (tp + tn) / (tp + fp + fn + tn)
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": round(prec,3), "recall": round(rec,3),
            "F1": round(f1,3), "akurasi": round(acc,3)}

y_prob = model.predict(X_test, verbose=0).ravel()
for thr in [0.2, 0.5, 0.8]:
    m = hitung_metrik(y_test, y_prob, thr)
    print(f"Threshold {thr}: {m}")

## 6. Diskusi

- Perhatikan bagaimana **recall** turun dan **precision** naik saat threshold dinaikkan.
- Akurasi terlihat tinggi karena mayoritas 'tidak hujan deras' — tetapi pada fenomena langka, F1/precision/recall lebih informatif.
- Bab 5 akan memperkenalkan CSI/FAR/POD/TS sesuai pedoman WMO; Bab 9 menerapkannya pada data stasiun BMKG.

## 7. Latihan Mini

1. Ubah `prob_deras` menjadi 0.5 (seimbang) — bandingkan metriknya.
2. Latih model multi-kelas (3 kelas intensitas) dan lihat confusion matrix.
3. Plot precision-recall untuk beberapa threshold.
4. Diskusikan: threshold mana yang Anda pilih jika false alarm mahal? Jika miss mahal?